# LM Studio 四模型批量 Li 评测

这个 notebook 与已完成的 CNC 批量评测隔离，复用相同的模型切换、smoke、generation 和 evaluation 流程。

运行分为两个阶段：

1. **Smoke 批量切换**：依次加载四个模型，用下雨导致洪水的句子确认模型可调用、思考关闭且能返回合法因果 JSON，然后立即卸载。
2. **正式批量评测**：手动设置 `RUN_BATCH_EVAL=True` 后，依次评估 Li 全部 786 条并分别保存报告。Qwen 使用 v10.1，Gemma 使用 v10.2。

第一轮保持 `ENABLE_RAG=False`。完成后如需运行第二轮，只把它改为 `True`；沿用 CNC validation 消融选定的家族配置：Qwen 使用 CNC KNN+Pattern k=3，Gemma 使用 CNC KNN+Pattern k=1。两个 Prompt 也会在预检时验证 RAG 槽位。


In [ ]:
# ===== 全局配置：运行前先检查本单元格 =====
import json
import logging
import sys
from collections.abc import Iterable, Iterator
from html import escape
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if 'master_thesis' not in sys.executable.lower():
    raise RuntimeError('请切换到 Master_thesis kernel，重启 kernel 后从第一格重新运行。')

from src.data_io import load_dataset
from src.eval_pipeline import EvalRunConfig, run_stream_eval
from src.generator import generate, parse_output
from src.llm_client import LLMClient
from src.prompt_builder import load_prompt_template
from src.retriever import resolve_rag_cache_paths

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
LOGGER = logging.getLogger('lmstudio_batch_li_test')
logging.getLogger('src.llm_client').setLevel(logging.WARNING)


def _notebook_progress(
    iterable: Iterable[Any],
    *,
    total: int,
    desc: str,
) -> Iterator[Any]:
    safe_desc = escape(desc)
    safe_total = max(total, 1)

    def render(completed: int) -> HTML:
        percent = min(completed / safe_total * 100, 100.0)
        return HTML(
            f"<div style='width:100%'>"
            f"<div>{safe_desc}: {completed}/{total} ({percent:.1f}%)</div>"
            f"<progress value='{completed}' max='{safe_total}' "
            f"style='width:100%;height:18px'></progress></div>"
        )

    handle = display(render(0), display_id=True)
    for completed, item in enumerate(iterable, 1):
        yield item
        if handle is not None:
            handle.update(render(completed))

LMSTUDIO_BASE_URL = 'http://127.0.0.1:1234/v1'
LMSTUDIO_API_KEY = 'lm-studio'
MODEL_LOAD_TIMEOUT = 1200
LLM_TIMEOUT = 600
LLM_RETRY_TIMES = 3
CONTEXT_LENGTH = 8192
MAX_TOKENS = 2048
TEMPERATURE = 0.0

DATASET_NAME = 'li'
EVAL_SAMPLE_N = None  # None = 评估当前 Li 数据集全部 786 条。
EVAL_PROGRESS_EVERY = 100
EVAL_MAX_WORKERS = 1
REPORT_DIR = Path('results') / 'eval_report' / 'lmstudio_batch_li'
RUN_BATCH_EVAL = False  # smoke 通过后手动改为 True。
REQUIRE_ALL_SMOKE_PASS = True

# 第一轮统一关闭 RAG；完成后只需把这个总开关改为 True 再运行第二轮。
ENABLE_RAG = False
LI_RAG_DATABASE = 'cnc'
LI_RAG_MODE = 'knn_pattern'
QWEN_RAG_TOP_K = 3
GEMMA_RAG_TOP_K = 1

# Qwen 使用既有最佳 Li prompt v10.1；Gemma 使用 validation 改善后的 v10.2。
# Gemma 31B 仍放在最后，避免它的偶发加载问题阻塞前三个模型。
MODEL_RUNS: list[dict[str, Any]] = [
    {
        'display_name': 'Qwen3.6 35B A3B',
        'model_key': 'qwen/qwen3.6-35b-a3b',
        'prompt_name': 'v10.1',
        'use_rag': ENABLE_RAG,
        'rag_database': LI_RAG_DATABASE,
        'rag_mode': LI_RAG_MODE,
        'rag_top_k': QWEN_RAG_TOP_K,
    },
    {
        'display_name': 'Qwen3.6 27B No Thinking',
        'model_key': 'local/qwen3.6-27b-no-thinking',
        'prompt_name': 'v10.1',
        'use_rag': ENABLE_RAG,
        'rag_database': LI_RAG_DATABASE,
        'rag_mode': LI_RAG_MODE,
        'rag_top_k': QWEN_RAG_TOP_K,
    },
    {
        'display_name': 'Gemma 4 26B A4B QAT',
        'model_key': 'google/gemma-4-26b-a4b-qat',
        'prompt_name': 'v10.2',
        'use_rag': ENABLE_RAG,
        'rag_database': LI_RAG_DATABASE,
        'rag_mode': LI_RAG_MODE,
        'rag_top_k': GEMMA_RAG_TOP_K,
    },
    {
        'display_name': 'Gemma 4 31B QAT',
        'model_key': 'google/gemma-4-31b-qat',
        'prompt_name': 'v10.2',
        'use_rag': ENABLE_RAG,
        'rag_database': LI_RAG_DATABASE,
        'rag_mode': LI_RAG_MODE,
        'rag_top_k': GEMMA_RAG_TOP_K,
    },
]

display(pd.DataFrame(MODEL_RUNS))


In [ ]:
# ===== LM Studio 管理 API 与四个 model key 预检（不加载模型） =====
manager = LLMClient(
    provider='lmstudio',
    base_url=LMSTUDIO_BASE_URL,
    model=MODEL_RUNS[0]['model_key'],
    api_key=LMSTUDIO_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    context_length=CONTEXT_LENGTH,
    reasoning='off',
    timeout=MODEL_LOAD_TIMEOUT,
    retry_times=LLM_RETRY_TIMES,
)

inventory = manager.list_local_models()
inventory_by_key = {str(item['key']): item for item in inventory if item.get('key')}
target_keys = [str(run['model_key']) for run in MODEL_RUNS]
missing_keys = [key for key in target_keys if key not in inventory_by_key]
if missing_keys:
    raise RuntimeError(f'LM Studio 中找不到目标模型：{missing_keys}')

inventory_rows: list[dict[str, Any]] = []
for run in MODEL_RUNS:
    item = inventory_by_key[str(run['model_key'])]
    reasoning = item.get('capabilities', {}).get('reasoning', {})
    allowed = reasoning.get('allowed_options', [])
    inventory_rows.append({
        'model': run['display_name'],
        'model_key': run['model_key'],
        'quantization': item.get('quantization', {}).get('name'),
        'params': item.get('params_string'),
        'reasoning_default': reasoning.get('default'),
        'reasoning_off_capability': (
            'advertised' if 'off' in allowed
            else 'not_supported' if allowed
            else 'smoke_required'
        ),
        'loaded_instances': len(item.get('loaded_instances', [])),
    })

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)
unsupported = inventory_df.loc[
    inventory_df['reasoning_off_capability'] == 'not_supported', 'model_key'
].tolist()
if unsupported:
    raise RuntimeError(f'以下模型不支持 reasoning=off：{unsupported}')
LOGGER.info('LM Studio 管理 API 与四个目标 model key 检查通过；smoke_required 将由实际生成验证。')

prompt_slot_counts = {
    prompt_name: load_prompt_template(prompt_name).count('{rag_examples}')
    for prompt_name in sorted({str(run['prompt_name']) for run in MODEL_RUNS})
}
invalid_prompt_slots = {
    prompt_name: count
    for prompt_name, count in prompt_slot_counts.items()
    if count != 1
}
if invalid_prompt_slots:
    raise RuntimeError(f'Li Prompt 的 RAG 槽位数量必须恰好为 1：{invalid_prompt_slots}')
LOGGER.info('Li Prompt RAG 槽位检查通过：%s', prompt_slot_counts)


In [ ]:
# ===== Smoke 批量切换辅助函数 =====
SMOKE_TEXT = 'Heavy rain caused flooding in the region.'
SMOKE_SYSTEM_PROMPT = (
    'Extract causal relations and output one strict JSON object only. '
    'Use schema {\"has_causal\": true, \"triples\": '    '[{\"cause\": {\"span\": \"...\"}, \"relation\": \"caused\", '    '\"effect\": {\"span\": \"...\"}}]}. '    'Do not output reasoning, Markdown, or any text outside the JSON.'
)


def _message_text(payload: dict[str, Any]) -> str:
    return '\n'.join(
        str(item.get('content', ''))
        for item in payload.get('output', [])
        if isinstance(item, dict) and item.get('type') == 'message'
    ).strip()


def run_smoke_batch(
    model_runs: list[dict[str, Any]],
    lmstudio: LLMClient,
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for position, run in enumerate(model_runs, 1):
        row: dict[str, Any] = {
            'order': position,
            'model': run['display_name'],
            'model_key': run['model_key'],
            'load': False,
            'reasoning_tokens': None,
            'no_reasoning': False,
            'causal_json': False,
            'unload': False,
            'passed': False,
            'error': '',
        }
        LOGGER.info('[smoke %s/%s] 准备加载 %s', position, len(model_runs), run['model_key'])
        try:
            lmstudio.unload_all_models()
            load_result = lmstudio.load_model(
                str(run['model_key']),
                context_length=CONTEXT_LENGTH,
            )
            instance_id = str(load_result['instance_id'])
            row['instance_id'] = instance_id
            row['load_config'] = json.dumps(load_result.get('load_config', {}), ensure_ascii=False)
            row['load'] = load_result.get('status') == 'loaded'

            smoke_payload = lmstudio.chat_rest(
                SMOKE_TEXT,
                model=instance_id,
                system_prompt=SMOKE_SYSTEM_PROMPT,
                reasoning='off',
                max_output_tokens=512,
            )
            content = _message_text(smoke_payload)
            reasoning_items = [
                item
                for item in smoke_payload.get('output', [])
                if isinstance(item, dict) and item.get('type') == 'reasoning'
            ]
            reasoning_tokens = smoke_payload.get('stats', {}).get('reasoning_output_tokens')
            row['reasoning_tokens'] = reasoning_tokens
            row['no_reasoning'] = (
                reasoning_tokens == 0
                and not reasoning_items
                and '<think>' not in content.lower()
            )
            parsed = parse_output(content)
            row['causal_json'] = bool(parsed.get('has_causal')) and bool(parsed.get('triples'))
            row['raw_output'] = content
            row['passed'] = bool(row['load'] and row['no_reasoning'] and row['causal_json'])
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
            LOGGER.exception('[smoke %s/%s] %s 失败', position, len(model_runs), run['model_key'])
        finally:
            try:
                lmstudio.unload_all_models()
                row['unload'] = True
            except Exception as unload_exc:
                row['passed'] = False
                unload_message = f'{type(unload_exc).__name__}: {unload_exc}'
                row['error'] = '; '.join(part for part in [row['error'], unload_message] if part)
                LOGGER.exception('[smoke %s/%s] 卸载失败', position, len(model_runs))
        results.append(row)
    return results


In [ ]:
# ===== 阶段 1：四模型逐个 load -> smoke -> unload =====
SMOKE_RESULTS = run_smoke_batch(MODEL_RUNS, manager)
smoke_df = pd.DataFrame(SMOKE_RESULTS)
display(smoke_df[[
    'order', 'model', 'model_key', 'load', 'reasoning_tokens',
    'no_reasoning', 'causal_json', 'unload', 'passed', 'error',
]])
LOGGER.info('Smoke 通过：%s/%s', int(smoke_df['passed'].sum()), len(smoke_df))


## 阶段 2：正式 Li 全数据集批量评测

先确认上面的四行 smoke 均为 `passed=True`。正式评测不会自动开始；回到全局配置，将 `RUN_BATCH_EVAL` 改为 `True`，再运行辅助函数和正式运行单元格。

第一轮 `ENABLE_RAG=False`。第二轮改为 `True` 后重新执行全局配置、预检、辅助函数和正式运行单元格；报告目录和文件名会按 RAG 状态区分。


In [ ]:
# ===== 正式批量评测辅助函数 =====
def _extra_body_for_run(run: dict[str, Any]) -> dict[str, Any]:
    return {'cache_prompt': False, **(run.get('llm_extra_body') or {})}


def _rag_paths_for_run(run: dict[str, Any]) -> tuple[Path | None, Path | None]:
    if not run['use_rag']:
        return None, None

    metadata_path, embeddings_path = resolve_rag_cache_paths(str(run['rag_database']))
    if DATASET_NAME == 'cnc_sft_test' and str(run['rag_database']) == 'cnc':
        sft_manifest = json.loads((PROJECT_ROOT / 'Data/CNC_sft/split_manifest.json').read_text(encoding='utf-8'))
        rag_manifest = json.loads((PROJECT_ROOT / 'RAG Database/cnc_split_manifest.json').read_text(encoding='utf-8'))
        test_ids = {str(sample_id) for sample_id in sft_manifest['split_ids']['test']}
        overlap = test_ids.intersection(str(sample_id) for sample_id in rag_manifest['support_ids'])
        if overlap:
            raise RuntimeError(
                f'拒绝在 cnc_sft_test 上使用现有 CNC RAG：support 与 test 有 {len(overlap)} 个精确 ID 重叠。'
            )
    return metadata_path, embeddings_path


def _load_eval_client(run: dict[str, Any]) -> tuple[str, LLMClient]:
    run_context_length = int(run.get('context_length', CONTEXT_LENGTH))
    run_parallel = run.get('parallel')
    run_offload_kv = bool(run.get('offload_kv_cache_to_gpu', True))
    load_result = manager.load_model(
        str(run['model_key']),
        context_length=run_context_length,
        parallel=None if run_parallel is None else int(run_parallel),
        offload_kv_cache_to_gpu=run_offload_kv,
    )
    load_config = load_result.get('load_config', {})
    actual_context = int(load_config.get('context_length', 0))
    actual_parallel = load_config.get('parallel')
    actual_offload_kv = load_config.get('offload_kv_cache_to_gpu')
    if (
        actual_context != run_context_length
        or (run_parallel is not None and int(actual_parallel or 0) != int(run_parallel))
        or actual_offload_kv is not run_offload_kv
    ):
        raise RuntimeError(
            '模型实际加载配置与目标不一致：'
            f'context={actual_context}/{run_context_length}, '
            f'parallel={actual_parallel}/{run_parallel}, '
            f'kv_cache_gpu={actual_offload_kv}/{run_offload_kv}。'
        )
    instance_id = str(load_result['instance_id'])
    client = LLMClient(
        provider='lmstudio',
        base_url=LMSTUDIO_BASE_URL,
        model=instance_id,
        api_key=LMSTUDIO_API_KEY,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        context_length=run_context_length,
        extra_body=_extra_body_for_run(run),
        timeout=LLM_TIMEOUT,
        retry_times=LLM_RETRY_TIMES,
    )
    return instance_id, client


def _summary_row(run: dict[str, Any], report: dict[str, Any]) -> dict[str, Any]:
    detection = report['detection']
    strict = report['extraction']['strict_token_f1']
    anchor = report['extraction']['anchor_window']
    return {
        'status': 'completed',
        'model': run['display_name'],
        'model_key': run['model_key'],
        'prompt': run['prompt_name'],
        'rag': 'off' if not run['use_rag'] else f"{run['rag_mode']}-k{run['rag_top_k']}",
        'temperature': TEMPERATURE,
        'context_length': int(run.get('context_length', CONTEXT_LENGTH)),
        'max_tokens': MAX_TOKENS,
        'parallel': run.get('parallel', 'model_default'),
        'reload_every_samples': int(run.get('reload_every_samples', 0) or 0),
        'cache_prompt': _extra_body_for_run(run)['cache_prompt'],
        'offload_kv_cache_to_gpu': bool(run.get('offload_kv_cache_to_gpu', True)),
        'reasoning': 'model_default_off',
        'n_samples': report['n_samples'],
        'n_causal_gold': report['n_causal_gold'],
        'n_causal_pred': report['n_causal_pred'],
        'detection_tp': detection['tp'],
        'detection_tn': detection['tn'],
        'detection_fp': detection['fp'],
        'detection_fn': detection['fn'],
        'all_n_eval_samples': strict['all_samples']['n_eval_samples'],
        'all_n_gold_triples': strict['all_samples']['n_gold_triples'],
        'all_n_pred_triples': strict['all_samples']['n_pred_triples'],
        'strict_all_tp': strict['all_samples']['tp'],
        'strict_all_fp': strict['all_samples']['fp'],
        'strict_all_fn': strict['all_samples']['fn'],
        'anchor_all_tp': anchor['all_samples']['tp'],
        'anchor_all_fp': anchor['all_samples']['fp'],
        'anchor_all_fn': anchor['all_samples']['fn'],
        'detected_n_eval_samples': strict['detected_only']['n_eval_samples'],
        'detected_n_gold_triples': strict['detected_only']['n_gold_triples'],
        'detected_n_pred_triples': strict['detected_only']['n_pred_triples'],
        'strict_detected_tp': strict['detected_only']['tp'],
        'strict_detected_fp': strict['detected_only']['fp'],
        'strict_detected_fn': strict['detected_only']['fn'],
        'anchor_detected_tp': anchor['detected_only']['tp'],
        'anchor_detected_fp': anchor['detected_only']['fp'],
        'anchor_detected_fn': anchor['detected_only']['fn'],
        'detection_precision': detection['precision'],
        'detection_recall': detection['recall'],
        'detection_f1': detection['f1'],
        'strict_all_precision': strict['all_samples']['precision'],
        'strict_all_recall': strict['all_samples']['recall'],
        'strict_all_f1': strict['all_samples']['f1'],
        'anchor_all_precision': anchor['all_samples']['precision'],
        'anchor_all_recall': anchor['all_samples']['recall'],
        'anchor_all_f1': anchor['all_samples']['f1'],
        'strict_detected_only_f1': strict['detected_only']['f1'],
        'anchor_detected_only_f1': anchor['detected_only']['f1'],
        'report_path': report.get('report_path', ''),
        'error': '',
    }


def run_formal_batch(
    model_runs: list[dict[str, Any]],
    *,
    existing_retrievers: dict[str, Any] | None = None,
    samples_override: list[dict[str, Any]] | None = None,
    require_all_smoke: bool = REQUIRE_ALL_SMOKE_PASS,
) -> list[dict[str, Any]]:
    if require_all_smoke:
        if 'SMOKE_RESULTS' not in globals():
            raise RuntimeError('请先运行四模型 smoke 单元格。')
        failed_smokes = [row['model_key'] for row in SMOKE_RESULTS if not row['passed']]
        if failed_smokes:
            raise RuntimeError(f'以下模型 smoke 未通过，正式评测未启动：{failed_smokes}')

    samples = (
        samples_override
        if samples_override is not None
        else load_dataset(DATASET_NAME, n=EVAL_SAMPLE_N)
    )
    rows: list[dict[str, Any]] = []
    for position, run in enumerate(model_runs, 1):
        LOGGER.info('[eval %s/%s] 开始 %s', position, len(model_runs), run['model_key'])
        try:
            manager.unload_all_models()
            metadata_path, embeddings_path = _rag_paths_for_run(run)
            run_id = str(run.get('run_id', run['model_key']))
            run_context_length = int(run.get('context_length', CONTEXT_LENGTH))
            reload_every_samples = int(run.get('reload_every_samples', 0) or 0)
            if reload_every_samples < 0:
                raise ValueError('reload_every_samples 不能为负数。')
            if reload_every_samples and EVAL_MAX_WORKERS != 1:
                raise RuntimeError('周期性模型重载要求 EVAL_MAX_WORKERS=1，以确保按样本计数。')
            instance_id, client = _load_eval_client(run)
            completed_samples = 0

            def generate_with_periodic_reload(**generator_kwargs: Any) -> dict[str, Any]:
                nonlocal instance_id, client, completed_samples
                if (
                    reload_every_samples
                    and completed_samples > 0
                    and completed_samples % reload_every_samples == 0
                ):
                    LOGGER.info(
                        '[eval] 已完成 %s 条，重新加载 %s 后继续。',
                        completed_samples,
                        run['model_key'],
                    )
                    manager.unload_all_models()
                    instance_id, client = _load_eval_client(run)
                generator_kwargs['client'] = client
                prediction = generate(**generator_kwargs)
                completed_samples += 1
                return prediction
            eval_config = EvalRunConfig(
                project_root=PROJECT_ROOT,
                model=instance_id,
                dataset=DATASET_NAME,
                prompt_name=str(run['prompt_name']),
                use_rag=bool(run['use_rag']),
                rag_mode=str(run['rag_mode']),
                rag_top_k=int(run['rag_top_k']),
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                primary_metric=str(run.get('primary_metric', 'anchor_window')),
                progress_every=EVAL_PROGRESS_EVERY,
                max_workers=EVAL_MAX_WORKERS,
                llm_provider='lmstudio',
                llm_base_url=LMSTUDIO_BASE_URL,
                context_length=run_context_length,
                reasoning_effort=None,
                llm_extra_body=_extra_body_for_run(run),
                api_key_source='lmstudio-default',
                save_report=True,
                report_dir=REPORT_DIR,
                report_detail_limit=200,
                report_detail_mode='errors',
                report_error_metric=str(run.get('primary_metric', 'anchor_window')),
                metadata_path=metadata_path,
                embeddings_path=embeddings_path,
            )
            report = run_stream_eval(
                samples=samples,
                label=str(run.get('label', f"{run['display_name']} Li full dataset")),
                client=client,
                config=eval_config,
                generator=generate_with_periodic_reload,
                existing_retriever=(existing_retrievers or {}).get(run_id),
                progress_factory=_notebook_progress,
                emit=LOGGER.info,
            )
            rows.append(_summary_row(run, report))
        except Exception as exc:
            LOGGER.exception('[eval %s/%s] %s 失败', position, len(model_runs), run['model_key'])
            rows.append({
                'status': 'failed',
                'model': run['display_name'],
                'model_key': run['model_key'],
                'prompt': run['prompt_name'],
                'rag': 'off' if not run['use_rag'] else f"{run['rag_mode']}-k{run['rag_top_k']}",
                'temperature': TEMPERATURE,
                'context_length': CONTEXT_LENGTH,
                'max_tokens': MAX_TOKENS,
                'parallel': run.get('parallel', 'model_default'),
                'reload_every_samples': int(run.get('reload_every_samples', 0) or 0),
                'cache_prompt': _extra_body_for_run(run)['cache_prompt'],
                'reasoning': 'model_default_off',
                'error': f'{type(exc).__name__}: {exc}',
            })
        finally:
            try:
                manager.unload_all_models()
            except Exception:
                LOGGER.exception('[eval %s/%s] 评测后卸载失败', position, len(model_runs))
    return rows


In [ ]:
# ===== 阶段 2：正式批量评测（默认不会运行） =====
if not RUN_BATCH_EVAL:
    LOGGER.warning('正式批量评测未启动。确认 smoke 后，将 RUN_BATCH_EVAL 改为 True。')
else:
    BATCH_RESULTS = run_formal_batch(MODEL_RUNS)
    batch_summary_df = pd.DataFrame(BATCH_RESULTS)
    display(batch_summary_df)
    batch_mode = 'family-rag-qwen-k3-gemma-k1' if ENABLE_RAG else 'rag-off'
    summary_path = PROJECT_ROOT / REPORT_DIR / f'li_full_{batch_mode}_summary.csv'
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    batch_summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
    LOGGER.info('Li 批量汇总已保存：%s', summary_path)
